In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import numpy as np
import jax.numpy as jnp

import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['text.usetex'] = True
mpl.rcParams['font.size'] = 10
mpl.rcParams['axes.grid'] = True
mpl.rcParams['axes.xmargin'] = 0
mpl.rcParams['lines.linewidth'] = 2
mpl.rcParams['legend.frameon'] = False
mpl.rcParams['savefig.bbox'] = 'tight'

import sys
sys.path.insert(0, "/home/joshua/PhD_year_1/jaxsp/Adding_stellar_masses")

import jaxsp as jsp

from scipy import constants as const

import matplotlib.animation as animation
from IPython.display import HTML


import Stellar_sim_funcs as SSF
import importlib
importlib.reload(SSF)


from scipy.interpolate import interp1d

from collections import defaultdict

from jaxsp.constants import h, om, hbar, Msun, GN, c, m22

from collections import defaultdict

In [ ]:
m22 = 1
u = jsp.set_schroedinger_units(m22)

G = GN.value * (u.from_cm**3) / (u.from_g * u.from_s**2)

In [ ]:
rho = jax.vmap(jsp.rho, in_axes=(0,None))


In [ ]:
cNFWtides_params = jnp.array([
    357964808.148399 * u.from_Msun, 
    25.690207, 
    0.407461, 
    0.012670 * u.from_Kpc, 
    1.857991 * u.from_Kpc, 
    3.729259
])
density_params = jsp.init_core_NFW_tides_params_from_sample(cNFWtides_params)
r99 = jsp.enclosing_radius(0.99, density_params) # radius that encloses 99% of mass
print(r99 * u.to_Kpc, "kpc")
density_params

In [ ]:
fig, ax = plt.subplots()
ax.xaxis.set_tick_params(labelsize=15)
ax.yaxis.set_tick_params(labelsize=15)
ax.set_xscale("log")
ax.set_yscale("log")
r = jnp.logspace(jnp.log10(100 * u.from_pc), jnp.log10(r99), 200)
ax.plot(r * u.to_Kpc, rho(r, density_params) * u.to_Msun/u.to_Kpc**3)
ax.set_ylabel(r"$\rho \;\;\mathrm{[M_\odot\;kpc^{-3}]}$", fontsize = 18)
ax.set_xlabel(r"$r \;\;\mathrm{[kpc]}$", fontsize = 18)
ax.set_title('DM density profile: NFW with cored center and tidal stripping effects', fontsize = 16)
plt.show()



In [ ]:
r = jnp.logspace(jnp.log10(1 * u.from_pc), jnp.log10(100), 10000)

rho_plot = rho(r, density_params)

M_enc = SSF.Enclosed_mass(r, rho_plot)

M_enc_jaxsp = jsp.core_nfw_tides_M(r, density_params)

total_mass_jaxsp = jsp.total_mass(density_params)
print("Total mass (JAXSP) =", total_mass_jaxsp * u.to_Msun * 10**-8, "M_sun")

fig, ax = plt.subplots()
ax.plot(r * u.to_Kpc, M_enc * u.to_Msun, label ='Custom implementation')
ax.plot(r * u.to_Kpc, M_enc_jaxsp * u.to_Msun, linestyle='--', label ='JAXSP implementation')
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylabel(r"$M_\mathrm{enc} \;\;\mathrm{[M_\odot]}$", fontsize = 18)
ax.set_xlabel(r"$r \;\;\mathrm{[kpc]}$", fontsize = 18)
ax.set_title('Enclosed mass profile', fontsize = 16)
plt.legend()
plt.show()

fig, ax = plt.subplots()
ax.plot(r * u.to_Kpc, M_enc / M_enc_jaxsp)
ax.set_xscale("log")
ax.set_ylabel(r"$M_\mathrm{enc, custom}/M_\mathrm{enc, jaxsp}$", fontsize = 18)
ax.set_xlabel(r"$r \;\;\mathrm{[kpc]}$", fontsize = 18)
ax.set_title('Enclosed mass profile comparison', fontsize = 16)
plt.show()

total_mass = M_enc[-1]
print("Total mass =", total_mass * u.to_Msun * 10**-8, "M_sun")

M_fraction = M_enc / total_mass_jaxsp

mask = M_fraction > 0.99
r_99 = r[mask][0]

fig, ax = plt.subplots()
ax.plot(r * u.to_Kpc, M_fraction)
ax.axvline(r_99 * u.to_Kpc, 0, 1, color='red', linestyle='--', label=r'$r_{99}$')
ax.set_xscale("log")
#ax.set_yscale("log")
ax.set_ylabel(r"$M_\mathrm{enc, custom}/M_\mathrm{total}$", fontsize = 18)
ax.set_xlabel(r"$r \;\;\mathrm{[kpc]}$", fontsize = 18)
ax.set_title('Enclosed mass fraction profile', fontsize = 16)
plt.show()

print(r_99 * u.to_Kpc, "kpc using my enclosed mass")

M_fraction_jaxsp = M_enc_jaxsp / total_mass_jaxsp

mask = M_fraction_jaxsp > 0.99
r_99_jaxsp = r[mask][0]
print(r_99_jaxsp * u.to_Kpc, "kpc using jaxsp")

fig, ax = plt.subplots()
ax.plot(r * u.to_Kpc, M_fraction_jaxsp)
ax.axvline(r_99 * u.to_Kpc, 0, 1, color='red', linestyle='--', label=r'$r_{99}$')
ax.set_xscale("log")
#ax.set_yscale("log")
ax.set_ylabel(r"$M_\mathrm{enc, jaxsp}/M_\mathrm{total}$", fontsize = 18)
ax.set_xlabel(r"$r \;\;\mathrm{[kpc]}$", fontsize = 18)
ax.set_title('Enclosed mass fraction profile', fontsize = 16)
plt.show()


# Total_mass_jaxsp is real total mass as mass_enclosed[-1] only goes out to r_max

# $\therefore$ $r_{99} = 440$ Kpc